# 02 — Rol 2: Clustering por partición

**Juan Esteban Ocampo (286388)**

Identificamos las zonas operativas reales de UrbanMove (K-means) y separamos la demanda
estructural de la esporádica (DBSCAN). Espacio de trabajo: coordenadas de recogida proyectadas
a kilómetros `(px, py)`, **sin escalar** — así ε de DBSCAN se lee directamente en metros.

Nota de reproducibilidad: aquí se evalúan los valores de K más informativos de la selección
(K=2, 4, 5, 9). El barrido completo (K=2..20) está documentado en la bitácora metodológica;
tarda varios minutos adicionales y no cambia la decisión.

In [1]:
import pandas as pd, numpy as np, importlib.util
spec = importlib.util.spec_from_file_location("config", "config.py")
cfgmod = importlib.util.module_from_spec(spec); spec.loader.exec_module(cfgmod)
SEMILLA, R, LAT_C, LON_C, LADO = cfgmod.SEMILLA, cfgmod.R_TIERRA_KM, cfgmod.LAT_C, cfgmod.LON_C, cfgmod.LADO_ZONA_KM

m = pd.read_parquet('muestra_50k.parquet')
cl = np.radians(LAT_C)
m['px'] = np.radians(m.pickup_longitude-LON_C)*R*np.cos(cl)
m['py'] = np.radians(m.pickup_latitude-LAT_C)*R
X = m[['px','py']].values
print(f"n = {len(m):,} | espacio: (px, py) en km, centro ({LAT_C}, {LON_C})")


n = 50,004 | espacio: (px, py) en km, centro (40.751, -73.9737)


## 1. K-means — selección de K

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

resultados = []
for K in [2, 4, 5, 9]:
    km = KMeans(K, init='k-means++', n_init=10, random_state=SEMILLA).fit(X)
    lab = km.labels_
    sil = silhouette_score(X, lab)
    db  = davies_bouldin_score(X, lab)
    aris = [adjusted_rand_score(lab, KMeans(K, n_init=10, random_state=SEMILLA+s).fit_predict(X)) for s in range(1, 4)]
    resultados.append(dict(K=K, silueta=round(sil,3), davies_bouldin=round(db,3), ari_min=round(min(aris),3)))
    print(resultados[-1])


{'K': 2, 'silueta': 0.809, 'davies_bouldin': 0.178, 'ari_min': 1.0}


{'K': 4, 'silueta': 0.492, 'davies_bouldin': 0.502, 'ari_min': 0.998}


{'K': 5, 'silueta': 0.452, 'davies_bouldin': 0.561, 'ari_min': 0.996}


{'K': 9, 'silueta': 0.416, 'davies_bouldin': 0.692, 'ari_min': 0.523}


In [3]:
# K=5: el codo de la curva de inercia y la escala operativa coherente con zonas de 1 km (Etapa 6)
km5 = KMeans(5, init='k-means++', n_init=10, random_state=SEMILLA).fit(X)
orden = np.argsort(-np.bincount(km5.labels_))
m['km5'] = pd.Series(km5.labels_).map({old: new for new, old in enumerate(orden)}).values
nombres = {0:'Midtown', 1:'Downtown', 2:'Upper Manhattan', 3:'LaGuardia', 4:'JFK'}
for k in range(5):
    print(f"KM{k} {nombres[k]:16s}: {(m.km5==k).mean()*100:5.2f}%")


KM0 Midtown         : 44.08%
KM1 Downtown        : 27.58%
KM2 Upper Manhattan : 23.11%
KM3 LaGuardia       :  3.06%
KM4 JFK             :  2.17%


## 2. DBSCAN — tratamiento del ruido

In [4]:
from sklearn.cluster import DBSCAN

# eps=418m viene de la rodilla de la gráfica k-distancia con min_samples=20 (ver bitácora)
db = DBSCAN(eps=0.418, min_samples=20).fit_predict(X)
m['db'] = db
m['ruido'] = (db == -1).astype(int)

f = len(m) / 1_438_943
print(f"Clusters encontrados: {db.max()+1}")
print(f"Ruido: {m.ruido.mean()*100:.3f}% ({m.ruido.sum()} viajes ≈ {m.ruido.sum()/f/182:.0f} recogidas reales/día)")

ari = adjusted_rand_score(m.km5, m.db)
print(f"\nARI K-means vs DBSCAN: {ari:.3f} (particiones casi opuestas, por diseño)")
print(f"% de recogidas en el cluster principal de DBSCAN (masa continua de Manhattan): "
      f"{(m.db == pd.Series(m.db).value_counts().index[0]).mean()*100:.2f}%")


Clusters encontrados: 10
Ruido: 1.034% (517 viajes ≈ 82 recogidas reales/día)

ARI K-means vs DBSCAN: 0.108 (particiones casi opuestas, por diseño)
% de recogidas en el cluster principal de DBSCAN (masa continua de Manhattan): 92.30%


In [5]:
# Caracterización del ruido: frecuencia real de recogidas alrededor de un punto de ruido
from sklearn.neighbors import NearestNeighbors
R_idx = m.ruido == 1
nn = NearestNeighbors(radius=0.418).fit(X)
vecinos = np.array([len(v)-1 for v in nn.radius_neighbors(m.loc[R_idx, ['px','py']].values, return_distance=False)])
tasa_dia = np.median(vecinos) / f / 182
print(f"Vecinos medianos en 418 m alrededor de un punto de ruido: {np.median(vecinos):.0f} en la muestra "
      f"-> {tasa_dia:.2f} recogidas reales/día -> una recogida cada {24/max(tasa_dia,0.01):.1f} horas")


Vecinos medianos en 418 m alrededor de un punto de ruido: 4 en la muestra -> 0.63 recogidas reales/día -> una recogida cada 37.9 horas


In [6]:
# Guardar etiquetas para la integración (notebook 04)
m[['id', 'km5', 'db', 'ruido']].to_parquet('rol2_etiquetas.parquet', index=False)
print("Guardado: rol2_etiquetas.parquet")


Guardado: rol2_etiquetas.parquet


## Resumen para la integración (Rol 4)

- K=5 (Midtown, Downtown, Upper Manhattan, LaGuardia, JFK) — descartado K=2 (trivial: solo aísla
  JFK) y K=9 (no reproducible entre semillas).
- DBSCAN confirma que Manhattan es una masa de densidad continua (>92% de las recogidas en un
  solo cluster): las zonas de K-means son cortes operativos, no fronteras naturales.
- El ruido (≈1% de la demanda) no debe cubrirse con flota fija: una recogida cada ~38 horas por
  punto no justifica posicionamiento permanente.